# Pierce the VEIL — Reconstruction Submission

**Tracks targeted:** Full Reconstruction, Partial Signal Recovery, Attack Strategy & Analysis, Best Write-Up.

## Methodology in one paragraph
We treat the published one-dimensional values as the only observable channel from a black-box encoder `E : R^D -> R`. Without paired `(x, z)` data the inverse is fundamentally ill-posed. We (1) forensically analyse the latent stream (basic stats, quantisation, modality, per-bit mantissa entropy), (2) vote on `D` anchored to a bank/credit domain prior, (3) score three candidate decoders (linear projection, mantissa bit-packing, mixed-radix digit packing) by self-consistency between the first principal component of the reconstruction and the latent, and (4) map the chosen reconstruction onto a published bank-feature marginal prior so output ranges are realistic. The full write-up establishes that per-column rank correlation is upper-bounded by `max_j |corr(z, x_j)|` and that this ceiling is matched by a trivial `D`-fold replication of `z` — see `writeup/writeup.md`.

## Compliance
* Deterministic (`SEED = 42`).
* No internet access required.
* Pure-CPU `numpy / scipy / scikit-learn`.
* Permutation-equivariant by construction.
* Outputs guaranteed finite.

## 1. Determinism and output sanitisation

In [1]:
import os
import random
import warnings
import numpy as np
from scipy import stats
from sklearn.exceptions import ConvergenceWarning
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

SEED = 42

def set_determinism(seed: int = SEED) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError:
        pass

def sanitize_output(x):
    x = np.asarray(x, dtype=np.float64)
    x = np.nan_to_num(x, nan=0.0, posinf=1e9, neginf=-1e9)
    return x

set_determinism()

## 2. Forensic analysis and dimensionality vote

In [2]:
DOMAIN_D_PRIOR = 16  # median D across UCI German Credit, Bank Marketing, LendingClub-style schemas

def basic_stats(z):
    z = np.asarray(z, dtype=np.float64).ravel()
    return {
        'n': int(z.size), 'min': float(z.min()), 'max': float(z.max()),
        'mean': float(z.mean()), 'std': float(z.std()),
        'n_unique': int(np.unique(z).size),
        'frac_unique': float(np.unique(z).size / z.size),
    }

def quantization_probe(z):
    z = np.sort(np.asarray(z, dtype=np.float64).ravel())
    gaps = np.diff(z)
    gaps = gaps[gaps > 1e-15]
    if gaps.size == 0:
        return {'looks_quantized': True, 'median_gap': 0.0}
    step = np.median(gaps)
    residual = np.mod(z - z[0], step)
    residual = np.minimum(residual, step - residual)
    return {
        'median_gap': float(step),
        'looks_quantized': bool(np.mean(residual / step) < 0.05),
    }

def modality_probe(z, k_max=12):
    z = np.asarray(z, dtype=np.float64).reshape(-1, 1)
    n_unique = int(np.unique(z).size)
    k_max = max(1, min(k_max, n_unique - 1))
    results = []
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        for k in range(1, k_max + 1):
            try:
                gmm = GaussianMixture(n_components=k, random_state=42, max_iter=200, n_init=2)
                gmm.fit(z)
                results.append((k, float(gmm.bic(z))))
            except Exception:
                continue
    if not results:
        return {'best_k': 1}
    return {'best_k': int(min(results, key=lambda r: r[1])[0])}

def float_bit_probe(z):
    z = np.asarray(z, dtype=np.float64).ravel()
    bits = z.view(np.uint64)
    mantissa = bits & ((1 << 52) - 1)
    H = []
    for i in range(52):
        b = ((mantissa >> i) & 1).astype(np.float64)
        p = b.mean()
        if p <= 0 or p >= 1:
            H.append(0.0)
        else:
            H.append(float(-p * np.log2(p) - (1 - p) * np.log2(1 - p)))
    return {'mantissa_entropy_mean': float(np.mean(H))}

def dimensionality_vote(z):
    n_unique = int(np.unique(z).size)
    if n_unique < 4:
        return 4
    D = DOMAIN_D_PRIOR
    bits = float_bit_probe(z)
    quant = quantization_probe(z)
    if bits['mantissa_entropy_mean'] > 0.99 and quant.get('looks_quantized', False):
        D += 4
    if n_unique < 64:
        D -= 4
    mod_k = modality_probe(z)['best_k']
    if mod_k >= 4:
        D = max(D, mod_k)
    return int(np.clip(D, 4, 32))

## 3. Hypothesis-driven decoders

In [3]:
def hypothesis_linear_projection(z, D):
    z = np.asarray(z, dtype=np.float64).ravel()
    N = z.size
    ranks = stats.rankdata(z, method='average') / (N + 1)
    cols = []
    for j in range(D):
        phase = 2.0 * np.pi * (j + 1) / (D + 1)
        cols.append(stats.norm.ppf(np.clip(ranks, 1e-6, 1 - 1e-6)) * np.cos(phase)
                    + (z - z.mean()) / (z.std() + 1e-12) * np.sin(phase))
    return np.stack(cols, axis=1)

def hypothesis_bitpack(z, D):
    z = np.asarray(z, dtype=np.float64).ravel()
    bits = z.view(np.uint64)
    mantissa = bits & ((1 << 52) - 1)
    bits_per_feature = max(1, 52 // D)
    cols = []
    for j in range(D):
        shift = j * bits_per_feature
        mask = (1 << bits_per_feature) - 1
        slice_ = ((mantissa >> shift) & mask).astype(np.float64)
        slice_ /= float(mask)
        cols.append(slice_)
    return np.stack(cols, axis=1)

def hypothesis_mixed_radix(z, D, base=10):
    z = np.asarray(z, dtype=np.float64).ravel()
    z_scaled = (z - z.min()) / (z.max() - z.min() + 1e-12)
    cols = []
    cur = z_scaled.copy()
    for _ in range(D):
        cur = cur * base
        digit = np.floor(cur)
        cur = cur - digit
        cols.append(digit / (base - 1))
    return np.stack(cols, axis=1)

def self_consistency_score(X_hat, z):
    z = np.asarray(z, dtype=np.float64).ravel()
    if X_hat.shape[1] < 2:
        return -np.inf
    pc1 = PCA(n_components=1, random_state=42).fit_transform(X_hat).ravel()
    return float(abs(np.corrcoef(pc1, z)[0, 1]))

def pick_best_hypothesis(z, D):
    candidates = {
        'linear': hypothesis_linear_projection(z, D),
        'bitpack': hypothesis_bitpack(z, D),
        'mixed_radix': hypothesis_mixed_radix(z, D),
    }
    scored = [(name, X, self_consistency_score(X, z)) for name, X in candidates.items()]
    scored.sort(key=lambda r: r[2], reverse=True)
    return scored[0][1], scored[0][0]

## 4. Bank/credit domain prior
Published marginals from UCI German Credit and UCI Bank Marketing.

In [4]:
BANK_FEATURE_PRIOR = [
    ('age',             42.0,  12.0,  (18,    95)),
    ('income',          55000, 30000, (0,     500000)),
    ('balance',         2500,  4500,  (-5000, 80000)),
    ('loan_amount',     12000, 9000,  (0,     300000)),
    ('loan_term',       36,    18,    (6,     360)),
    ('interest_rate',   7.5,   4.0,   (0,     35)),
    ('credit_score',    680,   90,    (300,   850)),
    ('debt_ratio',      0.35,  0.20,  (0,     2.0)),
    ('n_open_accounts', 8,     5,     (0,     40)),
    ('delinquencies',   0.3,   1.0,   (0,     20)),
    ('inquiries_6mo',   1.0,   1.5,   (0,     20)),
    ('employment_yrs',  7.5,   7.0,   (0,     50)),
    ('home_owner',      0.5,   0.5,   (0,     1)),
    ('has_default',     0.1,   0.3,   (0,     1)),
    ('region_code',     5.0,   3.0,   (0,     20)),
    ('product_type',    2.0,   1.0,   (0,     8)),
]

def apply_prior(X_hat):
    X_hat = np.asarray(X_hat, dtype=np.float64)
    N, D = X_hat.shape
    out = np.empty_like(X_hat)
    for j in range(D):
        col = X_hat[:, j]
        col = (col - col.mean()) / (col.std() + 1e-12)
        if j < len(BANK_FEATURE_PRIOR):
            _, mean, std, (lo, hi) = BANK_FEATURE_PRIOR[j]
            out[:, j] = np.clip(col * std + mean, lo, hi)
        else:
            out[:, j] = col
    return out

## 5. The required `reconstruct()` entry point

In [5]:
def reconstruct(public_latents, hidden_latents, metadata=None):
    """Required interface. Returns X_hat of shape (N_hidden, D_hat)."""
    set_determinism()
    pub = np.asarray(public_latents, dtype=np.float64).ravel()
    hid = np.asarray(hidden_latents, dtype=np.float64).ravel()
    combined = np.concatenate([pub, hid]) if pub.size > 0 else hid
    if isinstance(metadata, dict) and 'D' in metadata:
        D = int(metadata['D'])
    else:
        D = dimensionality_vote(combined)
    X_raw, _winner = pick_best_hypothesis(hid, D)
    X_hat = apply_prior(X_raw)
    return sanitize_output(X_hat)

# Quick contract self-test
_rng = np.random.default_rng(0)
_pub = _rng.normal(size=128)
_hid = _rng.normal(size=32)
_X = reconstruct(_pub, _hid)
assert _X.shape[0] == 32 and _X.ndim == 2 and np.all(np.isfinite(_X))
_perm = _rng.permutation(32)
_Xp = reconstruct(_pub, _hid[_perm])
assert np.allclose(_Xp, _X[_perm], rtol=1e-10, atol=1e-10), 'permutation equivariance failed'
print('contract checks passed; X_hat shape:', _X.shape)

contract checks passed; X_hat shape: (32, 16)


## 6. Load competition data and run
Auto-detects Kaggle vs local; falls back to the synthetic harness offline.

In [6]:
import glob

ON_KAGGLE = os.path.exists('/kaggle/input')

def _load_first(patterns):
    for p in patterns:
        for hit in sorted(glob.glob(p)):
            if hit.endswith('.npy'):
                return np.load(hit), hit
            if hit.endswith('.csv'):
                try:
                    arr = np.loadtxt(hit, delimiter=',', skiprows=1)
                except Exception:
                    arr = np.loadtxt(hit, delimiter=',')
                return arr, hit
    return None, None

if ON_KAGGLE:
    pub, pub_path = _load_first([
        '/kaggle/input/*/public*.npy', '/kaggle/input/*/public*.csv',
        '/kaggle/input/*/train*.npy', '/kaggle/input/*/train*.csv',
    ])
    hid, hid_path = _load_first([
        '/kaggle/input/*/hidden*.npy', '/kaggle/input/*/hidden*.csv',
        '/kaggle/input/*/test*.npy', '/kaggle/input/*/test*.csv',
    ])
    if pub is None:
        pub = np.array([])
    print('Kaggle paths:', pub_path, '|', hid_path)
else:
    pub = np.load('data/synthetic/public_latents.npy')
    hid = np.load('data/synthetic/hidden_latents.npy')
    print('Local synthetic data loaded.')

pub = np.asarray(pub, dtype=np.float64).ravel()
hid = np.asarray(hid, dtype=np.float64).ravel()
print('public_latents shape:', pub.shape, 'hidden_latents shape:', hid.shape)

X_hat = reconstruct(pub, hid)
print('X_hat shape:', X_hat.shape, 'finite:', bool(np.all(np.isfinite(X_hat))))

Local synthetic data loaded.
public_latents shape: (3584,) hidden_latents shape: (512,)
X_hat shape: (512, 16) finite: True


## 7. Save outputs

In [7]:
OUT_DIR = '/kaggle/working' if ON_KAGGLE else '.'
np.save(os.path.join(OUT_DIR, 'X_hat.npy'), X_hat)
header = ','.join([f'f{j}' for j in range(X_hat.shape[1])])
np.savetxt(os.path.join(OUT_DIR, 'X_hat.csv'), X_hat, delimiter=',', header=header, comments='')
print('Wrote X_hat.npy and X_hat.csv to', OUT_DIR)

Wrote X_hat.npy and X_hat.csv to .


## 8. Forensic report (for the write-up)

In [8]:
report = {
    'basic': basic_stats(pub) if pub.size > 0 else basic_stats(hid),
    'quantization': quantization_probe(pub) if pub.size > 0 else quantization_probe(hid),
    'modality_best_k': modality_probe(pub)['best_k'] if pub.size > 0 else modality_probe(hid)['best_k'],
    'mantissa_entropy_mean': float_bit_probe(pub)['mantissa_entropy_mean'] if pub.size > 0 else float_bit_probe(hid)['mantissa_entropy_mean'],
    'D_estimate': dimensionality_vote(np.concatenate([pub, hid])) if pub.size > 0 else dimensionality_vote(hid),
}
print('Forensic report:')
for k, v in report.items():
    print(f'  {k}: {v}')

Forensic report:
  basic: {'n': 3584, 'min': -5.23705855856025, 'max': 3.067871770937102, 'mean': -0.00032703419248565824, 'std': 1.0129778409867272, 'n_unique': 3584, 'frac_unique': 1.0}
  quantization: {'median_gap': 0.0007228847833984364, 'looks_quantized': False}
  modality_best_k: 1
  mantissa_entropy_mean: 0.9989270934363769
  D_estimate: 16
